# ADB Indonesia 2005–2025 — RRP & PCR document collector

**Input:** 96 ADB project IDs from your supplied inventory, embedded in this notebook. **Outputs:** candidate document index, reviewed downloads, verification hashes, error log and ZIP archive.

**Important:** This notebook has not been tested against ADB from Google Colab. ADB may block requests or change its HTML. Discovery is **candidate generation**, not proof of correct project/loan/subprogram matching. For research integrity, only rows marked `approved=Yes` are downloaded. Some projects have no published PCR, and a single program PCR can cover multiple subprograms.

Run cells top to bottom using **Runtime → Run all**, but pause at the review cell before approving any files. The notebook uses official `adb.org` domains only, avoids bypass techniques, and stops on HTTP 403/429 during discovery or download.

## 1. Install dependencies

In [ ]:
!pip -q install "requests>=2.31,<3" "beautifulsoup4>=4.12,<5"

## 2. Create embedded project inventory and downloader

In [ ]:
from pathlib import Path
import csv, io
WORK = Path('/content/ADB_Indonesia_Downloader')
WORK.mkdir(parents=True, exist_ok=True)
PROJECTS_CSV = 'ADB project ID,Project title (as listed),ADB project documents page,RRP PDF downloaded,Completion report PDF downloaded,Review status\n38354-013,State Accountability Revitalization Project,https://www.adb.org/projects/38354-013/main,No,No,Document links not yet verified\n48207-002,Program (Subprogram 2),https://www.adb.org/projects/48207-002/main,No,No,Document links not yet verified\n50297-001,Project,https://www.adb.org/projects/50297-001/main,No,No,Document links not yet verified\n50395-006,Growth Project,https://www.adb.org/projects/50395-006/main,No,No,Document links not yet verified\n51114-001,(Phase 2),https://www.adb.org/projects/51114-001/main,No,No,Document links not yet verified\n51157-001,Flood Management in North Java Project,https://www.adb.org/projects/51157-001/main,No,No,Document links not yet verified\n52282-001,Geothermal Power Generation Project,https://www.adb.org/projects/52282-001/main,No,No,Document links not yet verified\n52316-001,Reconstruction,https://www.adb.org/projects/52316-001/main,No,No,Document links not yet verified\n52332-001,Innovation Project,https://www.adb.org/projects/52332-001/main,No,No,Document links not yet verified\n53211-003,3,https://www.adb.org/projects/53211-003/main,No,No,Document links not yet verified\n53220-001,Program - Western and Central Java,https://www.adb.org/projects/53220-001/main,No,No,Document links not yet verified\n54152-001,Green Finance Facility (Phase 1),https://www.adb.org/projects/54152-001/main,No,No,Document links not yet verified\n54224-001,Transformation Program,https://www.adb.org/projects/54224-001/main,No,No,Document links not yet verified\n54224-002,Project,https://www.adb.org/projects/54224-002/main,No,No,Document links not yet verified\n54256-001,Sector Project,https://www.adb.org/projects/54256-001/main,No,No,Document links not yet verified\n54428-001,Citywide Inclusive Sanitation Project,https://www.adb.org/projects/54428-001/main,No,No,Document links not yet verified\n55020-001,Aquaculture Project,https://www.adb.org/projects/55020-001/main,No,No,Document links not yet verified\n55063-001,Parks Project,https://www.adb.org/projects/55063-001/main,No,No,Document links not yet verified\n56257-001,Subprogram 1,https://www.adb.org/projects/56257-001/main,No,No,Document links not yet verified\n57074-001,Markets - Phase 2 Preparation Project,https://www.adb.org/projects/57074-001/main,No,No,Document links not yet verified\n52282-003,Additional Financing),https://www.adb.org/projects/52282-003/main,No,No,Document links not yet verified\n54461-003,Development Program (Subprogram 3),https://www.adb.org/projects/54461-003/main,No,No,Document links not yet verified\n56017-001,Trans South-South Java Road Project,https://www.adb.org/projects/56017-001/main,No,No,Document links not yet verified\n56140-001,Transition Program (Phase 1),https://www.adb.org/projects/56140-001/main,No,No,Document links not yet verified\n32507-013,and Social Security Reform Program II),https://www.adb.org/projects/32507-013/main,No,No,Document links not yet verified\n32507-023,Subprogram 2,https://www.adb.org/projects/32507-023/main,No,No,Document links not yet verified\n33409-013,Vocational Education Strengthening Project,https://www.adb.org/projects/33409-013/main,No,No,Document links not yet verified\n34100-013,Renewable Energy Development,https://www.adb.org/projects/34100-013/main,No,No,Document links not yet verified\n34152-013,Community Water Services and Health,https://www.adb.org/projects/34152-013/main,No,No,Document links not yet verified\n34159-013,Road Rehabilitation II,https://www.adb.org/projects/34159-013/main,No,No,Document links not yet verified\n35139-013,Power Transmission Improvement Sector,https://www.adb.org/projects/35139-013/main,No,No,Document links not yet verified\n35182-043,Sector Project,https://www.adb.org/projects/35182-043/main,No,No,Document links not yet verified\n35183-013,Food Security and Poverty Reduction,https://www.adb.org/projects/35183-013/main,No,No,Document links not yet verified\n36541-013,Reform Program,https://www.adb.org/projects/36541-013/main,No,No,Document links not yet verified\n36541-023,Reform (Project Loan),https://www.adb.org/projects/36541-023/main,No,No,Document links not yet verified\n37049-013,Investment Program,https://www.adb.org/projects/37049-013/main,No,No,Document links not yet verified\n37049-023,Management Investment Program - Project 1,https://www.adb.org/projects/37049-023/main,No,No,Document links not yet verified\n37475-013,Madrasah Education Development,https://www.adb.org/projects/37475-013/main,No,No,Document links not yet verified\n38117-013,Empowerment,https://www.adb.org/projects/38117-013/main,No,No,Document links not yet verified\n38264-013,Governance Reform Program - Subprogram 1,https://www.adb.org/projects/38264-013/main,No,No,Document links not yet verified\n38385-013,Mandiri Project II,https://www.adb.org/projects/38385-013/main,No,No,Document links not yet verified\n38479-034,Regional Roads Development Project,https://www.adb.org/projects/38479-034/main,No,No,Document links not yet verified\n39071-013,Health Project,https://www.adb.org/projects/39071-013/main,No,No,Document links not yet verified\n39597-013,Rural Infrastructure Support Project,https://www.adb.org/projects/39597-013/main,No,No,Document links not yet verified\n40003-013,Development Goals Acceleration Program,https://www.adb.org/projects/40003-013/main,No,No,Document links not yet verified\n40009-013,Infrastructure Project Development Facility,https://www.adb.org/projects/40009-013/main,No,No,Document links not yet verified\n40009-023,Program (Subprogram 1),https://www.adb.org/projects/40009-023/main,No,No,Document links not yet verified\n40009-033,Program (Subprogram 2),https://www.adb.org/projects/40009-033/main,No,No,Document links not yet verified\n40009-043,Program - Subprogram 3,https://www.adb.org/projects/40009-043/main,No,No,Document links not yet verified\n40061-013,Improvement Project,https://www.adb.org/projects/40061-013/main,No,No,Document links not yet verified\n40247-013,Project,https://www.adb.org/projects/40247-013/main,No,No,Document links not yet verified\n40272-013,Third Development Policy Support Program,https://www.adb.org/projects/40272-013/main,No,No,Document links not yet verified\n40273-013,Fourth Development Policy Support Program,https://www.adb.org/projects/40273-013/main,No,No,Document links not yet verified\n41074-013,Project,https://www.adb.org/projects/41074-013/main,No,No,Document links not yet verified\n42099-013,Polytechnics Education Development Project,https://www.adb.org/projects/42099-013/main,No,No,Document links not yet verified\n42109-013,Company Project,https://www.adb.org/projects/42109-013/main,No,No,Document links not yet verified\n42362-013,Crossing,https://www.adb.org/projects/42362-013/main,No,No,Document links not yet verified\n43009-013,Public Expenditure Support Facility Program,https://www.adb.org/projects/43009-013/main,No,No,Document links not yet verified\n43092-013,Fifth Development Policy Support Program,https://www.adb.org/projects/43092-013/main,No,No,Document links not yet verified\n43211-013,Sixth Development Policy Support Program,https://www.adb.org/projects/43211-013/main,No,No,Document links not yet verified\n43220-014,Management of Irrigation Program,https://www.adb.org/projects/43220-014/main,No,No,Document links not yet verified\n43251-025,Investment Project,https://www.adb.org/projects/43251-025/main,No,No,Document links not yet verified\n43255-013,Support to PNPM Mandiri Project,https://www.adb.org/projects/43255-013/main,No,No,Document links not yet verified\n44252-013,Program,https://www.adb.org/projects/44252-013/main,No,No,Document links not yet verified\n46093-001,Connectivity Program - Subprogram 1,https://www.adb.org/projects/46093-001/main,No,No,Document links not yet verified\n46093-004,"Connectivity Program, Subprogram 2",https://www.adb.org/projects/46093-004/main,No,No,Document links not yet verified\n46094-001,(Phase 2),https://www.adb.org/projects/46094-001/main,No,No,Document links not yet verified\n46239-001,Precautionary Financing Facility,https://www.adb.org/projects/46239-001/main,No,No,Document links not yet verified\n46421-001,Program-Coral Triangle Initiative Project,https://www.adb.org/projects/46421-001/main,No,No,Document links not yet verified\n48134-001,Acceleration Program-Subprogram 1,https://www.adb.org/projects/48134-001/main,No,No,Document links not yet verified\n48134-006,Acceleration Program (Subprogram 2),https://www.adb.org/projects/48134-006/main,No,No,Document links not yet verified\n48134-007,Acceleration Program (Subprogram 3),https://www.adb.org/projects/48134-007/main,No,No,Document links not yet verified\n48207-001,Program Subprogram 1,https://www.adb.org/projects/48207-001/main,No,No,Document links not yet verified\n48207-004,Program (Subprogram 3),https://www.adb.org/projects/48207-004/main,No,No,Document links not yet verified\n49043-001,(Subprogram 1),https://www.adb.org/projects/49043-001/main,No,No,Document links not yet verified\n49043-002,(Subprogram 2),https://www.adb.org/projects/49043-002/main,No,No,Document links not yet verified\n49043-003,(Subprogram 3),https://www.adb.org/projects/49043-003/main,No,No,Document links not yet verified\n49080-001,Program,https://www.adb.org/projects/49080-001/main,No,No,Document links not yet verified\n49141-001,Better Engineering Services Project,https://www.adb.org/projects/49141-001/main,No,No,Document links not yet verified\n50016-001,Electricity Grid Development Program,https://www.adb.org/projects/50016-001/main,No,No,Document links not yet verified\n50168-001,Program (Subprogram 1),https://www.adb.org/projects/50168-001/main,No,No,Document links not yet verified\n50168-002,Program (Subprogram 2),https://www.adb.org/projects/50168-002/main,No,No,Document links not yet verified\n50168-003,Program (Subprogram 3),https://www.adb.org/projects/50168-003/main,No,No,Document links not yet verified\n52218-001,Program (Subprogram 1),https://www.adb.org/projects/52218-001/main,No,No,Document links not yet verified\n52218-002,Program (Subprogram 2),https://www.adb.org/projects/52218-002/main,No,No,Document links not yet verified\n52324-001,Rehabilitation from Recent Disasters,https://www.adb.org/projects/52324-001/main,No,No,Document links not yet verified\n53211-001,1,https://www.adb.org/projects/53211-001/main,No,No,Document links not yet verified\n53211-002,2,https://www.adb.org/projects/53211-002/main,No,No,Document links not yet verified\n54117-001,Disaster Resilience Improvement Program,https://www.adb.org/projects/54117-001/main,No,No,Document links not yet verified\n54139-001,Support Program,https://www.adb.org/projects/54139-001/main,No,No,Document links not yet verified\n54461-001,Development Program (Subprogram 1),https://www.adb.org/projects/54461-001/main,No,No,Document links not yet verified\n54461-002,"Development Program, Subprogram 2",https://www.adb.org/projects/54461-002/main,No,No,Document links not yet verified\n55235-001,Subprogram 1,https://www.adb.org/projects/55235-001/main,No,No,Document links not yet verified\n56368-001,"Program, Subprogram 1",https://www.adb.org/projects/56368-001/main,No,No,Document links not yet verified\n57018-001,1,https://www.adb.org/projects/57018-001/main,No,No,Document links not yet verified\n58165-001,"Program, Subprogram 3",https://www.adb.org/projects/58165-001/main,No,No,Document links not yet verified\n'
DOWNLOADER_SOURCE = '#!/usr/bin/env python3\n"""ADB Indonesia sovereign-loan document discovery and reviewed PDF downloader.\nOnly official adb.org URLs are accepted. No login, CAPTCHA bypass, or proxy rotation.\n"""\nimport argparse, csv, hashlib, os, re, sys, time\nfrom pathlib import Path\nfrom urllib.parse import urljoin, urlparse, urldefrag\nimport requests\nfrom bs4 import BeautifulSoup\n\nROOT = Path(__file__).resolve().parent\nHEADERS = {\'User-Agent\': \'Mozilla/5.0 (compatible; ADBAcademicDocumentCollector/1.0; research download)\', \'Accept\': \'text/html,application/pdf;q=0.9,*/*;q=0.7\'}\nFIELDS = [\'project_id\',\'source_title_unverified\',\'document_type\',\'document_title\',\'source_page\',\'document_url\',\'match_reason\',\'approved\',\'status\',\'local_file\',\'sha256\',\'notes\']\nRRP = re.compile(r\'\\breport and recommendation of the president\\b|\\breport and recommendation\\b|\\brrp\\b\', re.I)\nPCR = re.compile(r\'\\bproject completion report\\b|\\bprogram completion report\\b|\\bcompletion report\\b|\\bpcr\\b\', re.I)\nOTHER = re.compile(r\'\\bvalidation report\\b|\\bevaluation\\b|\\bcompletion report validation\\b|\\btechnical assistance completion\\b\', re.I)\nID = re.compile(r\'^\\d{5}-\\d{3}$\')\n\ndef official(url):\n    p=urlparse(url)\n    return p.scheme==\'https\' and (p.hostname==\'adb.org\' or (p.hostname or \'\').endswith(\'.adb.org\'))\n\ndef session():\n    s=requests.Session();s.headers.update(HEADERS);return s\n\ndef fetch(s,url,delay):\n    if not official(url): raise ValueError(\'Non-ADB URL rejected\')\n    time.sleep(delay)\n    r=s.get(url,timeout=35,allow_redirects=True)\n    if not official(r.url): raise ValueError(\'Redirect outside ADB rejected\')\n    r.raise_for_status()\n    return r\n\ndef load_projects(path):\n    with open(path,encoding=\'utf-8-sig\',newline=\'\') as f: rows=list(csv.DictReader(f))\n    output=[];seen=set()\n    for row in rows:\n        pid=(row.get(\'ADB project ID\') or row.get(\'project_id\') or \'\').strip()\n        if not ID.fullmatch(pid) or pid in seen: continue\n        seen.add(pid)\n        output.append((pid,(row.get(\'Project title (as listed)\') or row.get(\'title\') or \'\').strip()))\n    if not output: raise ValueError(\'No valid project IDs in input CSV\')\n    return output\n\ndef save_rows(path,rows):\n    tmp=path.with_suffix(\'.tmp\')\n    with open(tmp,\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n        w=csv.DictWriter(f,fieldnames=FIELDS,extrasaction=\'ignore\');w.writeheader();w.writerows(rows)\n    tmp.replace(path)\n\ndef read_rows(path):\n    if not path.exists():return []\n    with open(path,newline=\'\',encoding=\'utf-8-sig\') as f:return list(csv.DictReader(f))\n\ndef doc_type(title,url):\n    # Use the visible title as primary evidence; filename suffixes are secondary.\n    if OTHER.search(title):return None\n    if RRP.search(title):return \'RRP\'\n    if PCR.search(title):return \'PCR\'\n    path=urlparse(url).path.lower()\n    if re.search(r\'[-_/]rrp(?:[-_.]|$)\',path):return \'RRP\'\n    if re.search(r\'[-_/]pcr(?:[-_.]|$)\',path):return \'PCR\'\n    return None\n\ndef extract_links(html,page_url,pid):\n    soup=BeautifulSoup(html,\'html.parser\')\n    out=[];seen=set()\n    for a in soup.select(\'a[href]\'):\n        href=a.get(\'href\',\'\').strip()\n        url=urldefrag(urljoin(page_url,href))[0]\n        if not official(url):continue\n        title=\' \'.join(a.stripped_strings).strip() or a.get(\'title\',\'\').strip()\n        aria=a.get(\'aria-label\',\'\')\n        title=title or aria\n        typ=doc_type(title,url)\n        if not typ:continue\n        # Keep candidates, but require human approval if title/ID evidence is weak.\n        key=(typ,url)\n        if key in seen:continue\n        seen.add(key)\n        match=\'Project page link; confirm project/loan coverage before approval\'\n        if pid in url:match=\'Project ID in document URL; confirm project/loan coverage before approval\'\n        out.append((typ,title or \'(filename classification only)\',url,match))\n    return out\n\ndef discover(args):\n    s=session();projects=load_projects(Path(args.input));out=Path(args.index)\n    existing=read_rows(out);bykey={(r[\'project_id\'],r[\'document_url\']):r for r in existing if r.get(\'document_url\')}\n    errors=[]\n    for i,(pid,title) in enumerate(projects,1):\n        pages=[f\'https://www.adb.org/projects/{pid}/main\',f\'https://www.adb.org/projects/{pid}/project-documents\']\n        found=0\n        for page in pages:\n            try:\n                r=fetch(s,page,args.delay)\n                if \'pdf\' in r.headers.get(\'content-type\',\'\').lower():continue\n                links=extract_links(r.text,r.url,pid)\n                for typ,doc_title,url,reason in links:\n                    key=(pid,url)\n                    if key not in bykey:\n                        bykey[key]=dict(project_id=pid,source_title_unverified=title,document_type=typ,document_title=doc_title,source_page=r.url,document_url=url,match_reason=reason,approved=\'\',status=\'NEEDS_REVIEW\',local_file=\'\',sha256=\'\',notes=\'Confirm document title, project ID, loan and subprogram\')\n                        found+=1\n            except (requests.RequestException,ValueError) as e:\n                errors.append((pid,page,str(e)))\n                print(f\'[{i}/{len(projects)}] {pid}: {type(e).__name__}: {e}\',file=sys.stderr)\n                if isinstance(e,requests.HTTPError) and e.response is not None and e.response.status_code in (403,429):\n                    print(\'Access restricted by ADB; stopping discovery. Use official site manually and paste URLs into index.\',file=sys.stderr)\n                    save_rows(out,list(bykey.values()))\n                    write_errors(Path(args.errors),errors)\n                    return\n        print(f\'[{i}/{len(projects)}] {pid}: {found} new candidates\')\n        save_rows(out,list(bykey.values()))\n    write_errors(Path(args.errors),errors)\n    print(f\'Discovery finished: {len(bykey)} candidates; review {out}\')\n\ndef write_errors(path,errors):\n    with open(path,\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n        w=csv.writer(f);w.writerow([\'project_id\',\'page\',\'error\']);w.writerows(errors)\n\ndef download(args):\n    index=Path(args.index);rows=read_rows(index)\n    if not rows:sys.exit(\'Index empty. Run discover, or add official ADB PDF URLs to the index first.\')\n    s=session();root=Path(args.output);root.mkdir(parents=True,exist_ok=True)\n    count=0\n    for i,row in enumerate(rows,1):\n        if row.get(\'approved\',\'\').strip().lower() not in (\'yes\',\'y\',\'true\'):continue\n        if row.get(\'status\')==\'DOWNLOADED\' and row.get(\'local_file\') and Path(row[\'local_file\']).is_file():continue\n        pid=row.get(\'project_id\',\'\');typ=row.get(\'document_type\',\'\')\n        url=row.get(\'document_url\',\'\').strip()\n        if not ID.fullmatch(pid) or typ not in (\'RRP\',\'PCR\') or not official(url):\n            row[\'status\']=\'INVALID_ENTRY\';row[\'notes\']=\'Invalid ID/type or non-ADB URL\';continue\n        if args.limit and count>=args.limit:break\n        destdir=root/pid;destdir.mkdir(exist_ok=True)\n        # Multiple RRPs/PCRs may exist for the same project; unique hash avoids overwrites.\n        suffix=hashlib.sha256(url.encode()).hexdigest()[:10]\n        dest=destdir/f\'{pid}_{typ}_{suffix}.pdf\'\n        temp=dest.with_suffix(\'.part\')\n        try:\n            time.sleep(args.delay)\n            with s.get(url,stream=True,timeout=60,allow_redirects=True) as r:\n                if not official(r.url):raise ValueError(\'Redirect outside ADB\')\n                r.raise_for_status()\n                content_type=r.headers.get(\'content-type\',\'\').lower()\n                if \'text/html\' in content_type or not (\'pdf\' in content_type or urlparse(r.url).path.lower().endswith(\'.pdf\')):\n                    raise ValueError(\'URL returned a document landing page, not a PDF; find its official PDF download link\')\n                h=hashlib.sha256();n=0;first=b\'\'\n                with open(temp,\'wb\') as f:\n                    for chunk in r.iter_content(65536):\n                        if not chunk:continue\n                        if not first:first=chunk[:8]\n                        f.write(chunk);h.update(chunk);n+=len(chunk)\n                if not first.startswith(b\'%PDF-\') or n<1024:raise ValueError(\'Invalid or empty PDF response\')\n                temp.replace(dest)\n                row.update(status=\'DOWNLOADED\',local_file=str(dest.resolve()),sha256=h.hexdigest(),notes=f\'PDF signature checked; {n} bytes\')\n                print(f\'[{i}/{len(rows)}] Saved {dest}\')\n        except (requests.RequestException,ValueError,OSError) as e:\n            row[\'status\']=\'FAILED\';row[\'notes\']=f\'{type(e).__name__}: {e}\'\n            print(f\'[{i}/{len(rows)}] FAILED {pid}: {e}\',file=sys.stderr)\n            if temp.exists():temp.unlink()\n            if isinstance(e,requests.HTTPError) and e.response is not None and e.response.status_code in (403,429):\n                print(\'ADB access restriction: stopping. Do not try to bypass it.\',file=sys.stderr)\n                save_rows(index,rows);return\n        count+=1\n        save_rows(index,rows)\n    print(f\'Download pass complete; inspect {index}\')\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'command\',choices=[\'discover\',\'download\'])\n    p.add_argument(\'--input\',default=str(ROOT/\'projects.csv\'))\n    p.add_argument(\'--index\',default=str(ROOT/\'document_index.csv\'))\n    p.add_argument(\'--errors\',default=str(ROOT/\'discovery_errors.csv\'))\n    p.add_argument(\'--output\',default=str(ROOT/\'ADB_PDFs\'))\n    p.add_argument(\'--delay\',type=float,default=3.0,help=\'Pause between ADB requests (seconds)\')\n    p.add_argument(\'--limit\',type=int,default=0,help=\'Maximum approved PDFs this pass (0=all)\')\n    args=p.parse_args()\n    if args.delay<1:sys.exit(\'Delay must be >= 1 second\')\n    (discover if args.command==\'discover\' else download)(args)\nif __name__==\'__main__\':main()\n'
(WORK/'projects.csv').write_text(PROJECTS_CSV, encoding='utf-8-sig')
(WORK/'adb_downloader.py').write_text(DOWNLOADER_SOURCE, encoding='utf-8')
records=list(csv.DictReader(io.StringIO(PROJECTS_CSV)))
print(f'Prepared {len(records)} project records in {WORK}')
assert len(records)==96, 'Unexpected number of project records'


## 3. Test network access to ADB (one request)

In [ ]:
import requests
TEST_URL='https://www.adb.org/sites/default/files/project-documents/43255/43255-013-pcr.pdf'
try:
    r=requests.get(TEST_URL,headers={'User-Agent':'Mozilla/5.0'},timeout=25,stream=True)
    print('HTTP status:',r.status_code,'Content-Type:',r.headers.get('Content-Type','unknown'))
    r.close()
    if r.status_code in (403,429):
        print('ADB restricted access. Do not bypass the restriction; use browser-based collection.')
except requests.RequestException as e:
    print('Network test failed:',type(e).__name__,str(e)[:300])


## 4. Discover candidate RRP/PCR links

This checks two official pages per project, pauses between requests, and records failures. For 96 projects, discovery may take 10+ minutes. It may find few or no candidates if ADB renders links dynamically or blocks access.

In [ ]:
import subprocess, sys
result=subprocess.run([sys.executable,str(WORK/'adb_downloader.py'),'discover','--delay','3'],cwd=WORK)
print('Discovery exit code:',result.returncode)


## 5. Inspect the candidate index (required human review)

Only approve a row after checking that the **document type, project ID, loan number, and subprogram** match. Do not assume a PDF with a matching project prefix covers every subprogram. The discovery script intentionally leaves `approved` blank.

In [ ]:
import pandas as pd
INDEX=WORK/'document_index.csv'
if INDEX.exists():
    df=pd.read_csv(INDEX,dtype=str).fillna('')
    print('Candidate rows:',len(df))
    display(df[['project_id','document_type','document_title','document_url','match_reason','approved','status']].head(50))
else:
    print('No index created; inspect discovery_errors.csv or ADB access.')


### Review and edit the index

Download `document_index.csv`, open it in Excel/Sheets, check each candidate's official URL, and set `approved` to `Yes` **only** for confirmed RRP/PCR matches. You may also add verified ADB PDF URLs as new rows with the same columns. Then upload the edited file in the next cell. This is deliberate: automatic approval could introduce serious classification errors into your thesis.

In [ ]:
from google.colab import files
if INDEX.exists():
    files.download(str(INDEX))
print('After reviewing the CSV locally, use the next cell to upload your edited copy.')


In [ ]:
# Run this cell ONLY after you have edited and saved document_index.csv.
from google.colab import files
uploaded=files.upload()
if 'document_index.csv' in uploaded:
    INDEX.write_bytes(uploaded['document_index.csv'])
    print('Reviewed index loaded:',len(pd.read_csv(INDEX)),'rows')
else:
    print('No reviewed index uploaded. Downloads will remain unapproved.')


## 6. Download approved PDFs and verify signatures / SHA-256

Start with five approved PDFs. The script checks `%PDF-`, minimum size, official domains, and writes SHA-256 hashes. If the test succeeds, run the next cell for all remaining approved rows.

In [ ]:
subprocess.run([sys.executable,str(WORK/'adb_downloader.py'),'download','--limit','5','--delay','3'],cwd=WORK)


In [ ]:
# Run only after reviewing the five-file test.
subprocess.run([sys.executable,str(WORK/'adb_downloader.py'),'download','--delay','3'],cwd=WORK)


## 7. Coverage audit — one row per project and document type

In [ ]:
from collections import Counter
ids=[r['ADB project ID'] for r in records]
rows=list(csv.DictReader(INDEX.open(encoding='utf-8-sig',newline=''))) if INDEX.exists() else []
audit=[]
for pid in ids:
    for typ in ('RRP','PCR'):
        matching=[r for r in rows if r.get('project_id')==pid and r.get('document_type')==typ]
        done=[r for r in matching if r.get('status')=='DOWNLOADED' and Path(r.get('local_file','')).is_file()]
        audit.append({'project_id':pid,'document_type':typ,'candidate_count':len(matching),
                      'downloaded_count':len(done),'audit_status':'DOWNLOADED' if done else ('REVIEW_OR_DOWNLOAD_PENDING' if matching else 'NOT_FOUND_BY_AUTOMATION')})
audit_file=WORK/'coverage_audit.csv'
with audit_file.open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0]));w.writeheader();w.writerows(audit)
print('Projects:',len(ids),'Expected audit rows:',len(audit))
print('Coverage status:',dict(Counter(r['audit_status'] for r in audit)))
display(pd.DataFrame(audit).head(20))


## 8. Export ZIP with PDFs, index, audit, errors, and downloader

The ZIP includes whatever was **actually downloaded**, which may be zero PDFs if ADB access is blocked or no links have been approved. The index and audit distinguish this clearly.

In [ ]:
import zipfile
ZIP=Path('/content/ADB_Indonesia_2005_2025_RRP_PCR_results.zip')
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in WORK.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.part':
            z.write(p,p.relative_to(WORK.parent))
print('Archive:',ZIP,'Size (MB):',round(ZIP.stat().st_size/1e6,2))
print('Verified PDF files:',len(list((WORK/'ADB_PDFs').rglob('*.pdf'))) if (WORK/'ADB_PDFs').exists() else 0)
files.download(str(ZIP))
